In [6]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-preparation").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64
Spark version: 4.1.2


26/06/27 02:24:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Drill -- Streaming Word Count

## Problem

Read a stream of text from a socket (localhost:11111) and perform a **live word count**:

- Split each incoming line into words (use `\W+` as the delimiter)
- Lowercase all words before counting
- Filter out empty strings
- Count occurrences of each word
- Output the **full word count table** (ordered by count descending) after each micro-batch

## Setup

Feed the poem via netcat:
```bash
# macOS/Linux
nc -lk 11111 < data/Safadi_poem.txt

# Or interactively type text:
nc -lk 11111
```

## Notes
- Use `complete` output mode (required for aggregations over the full stream)
- Use `orderBy(col("count").desc())` to see top words first

In [7]:
# Preview the poem to be streamed
with open("data/Safadi_poem.txt") as f:
    print(f.read())

Data was small but now data is big
Small grew to big and big is getting bigger
Volume, velocity, variety, veracity and value
So the elders have spoken
But ultimately insights are few and precise
So welcome to this new world
data data da ta da ta ta da



In [8]:
# Solution
input_stream = (spark.readStream
    .format("socket")
    .option("host", STREAM_HOST)
    .option("port", 11111)
    .load())

word_count = input_stream

# complete here
#
#
#

output = word_count.writeStream.format("console").outputMode("complete")

# NOTE: uncomment to run with netcat
query = output.start()
query.awaitTermination(30)
query.stop()

print("WordCount drill query defined")
print("Feed the poem: nc -lk 11111 < data/Safadi_poem.txt")


-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----+
|    word|count|
+--------+-----+
|    data|    4|
|     and|    3|
|      ta|    3|
|      da|    3|
|     big|    3|
|      is|    2|
|     but|    2|
|   small|    2|
|      so|    2|
|      to|    2|
|     few|    1|
|    grew|    1|
|     new|    1|
| variety|    1|
| welcome|    1|
|veracity|    1|
|     was|    1|
|  spoken|    1|
|insights|    1|
|  volume|    1|
+--------+-----+
only showing top 20 rows
WordCount drill query defined
Feed the poem: nc -lk 11111 < data/Safadi_poem.txt


In [9]:
spark.stop()